# 10a_GAN_DPS_hyperparameter_tuning
Tuning di `data_weight`, `step_size`, `noise_scale` per il ciclo DPS-adattato, con **Adam** come ottimizzatore per il vettore latente `z` (non discesa del gradiente a passo fisso -- quella dava risultati instabili, vincitore sempre al bordo della griglia). Il tuning gira a `N_ITERS=300`, lo stesso numero di iterazioni usato poi nella generazione finale, per evitare disallineamenti tra i due.

Su Colab con GPU. Un solo run, dall'inizio alla fine -- salva un solo file di risultati (`gan_dps_tuning/best_params_dps.json`), quello che `10_GAN_DPS_generation.ipynb` legge automaticamente.

## Metodologia
Griglia testata sul **validation set** (12 campioni, mai il test), criterio PSNR+SSIM combinato (50/50).

## Prerequisiti su Drive
`sinograms.zip`, `validation_resized.zip` (gia' presenti), `gan_checkpoints/generator_weights_best.pth` (il generatore senza decay, quello con la FID migliore: 314.77).

In [ ]:
# DA ESEGURIE SU COLAB PER GENERARE I PARAMTERI OTTIMIZZATI PER LA GAN_DPS

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

SINOGRAMS_ZIP = "/content/drive/MyDrive/sinograms.zip"
VALIDATION_RESIZED_ZIP = "/content/drive/MyDrive/validation_resized.zip"

LOCAL_SINOGRAMS_DIR = Path("/content/sinograms")
VALIDATION_DIR = Path("/content/validation_resized")

if not LOCAL_SINOGRAMS_DIR.exists():
    print("Estraggo i sinogrammi...")
    get_ipython().system('unzip -q "{SINOGRAMS_ZIP}" -d /content/')
else:
    print("Sinogrammi gia' estratti, salto.")

if not VALIDATION_DIR.exists():
    print("Estraggo validation_resized (ground truth)...")
    get_ipython().system('unzip -q "{VALIDATION_RESIZED_ZIP}" -d /content/')
else:
    print("validation_resized gia' estratto, salto.")

GENERATOR_WEIGHTS_PATH = Path("/content/drive/MyDrive/gan_checkpoints/generator_weights_best.pth")
assert GENERATOR_WEIGHTS_PATH.exists(), f"Non trovo {GENERATOR_WEIGHTS_PATH} -- il training del GAN e' finito?"

Mounted at /content/drive
Estraggo i sinogrammi...
Estraggo validation_resized (ground truth)...


In [2]:
get_ipython().system('pip install -q git+https://github.com/devangelista2/IPPy.git')
try:
    get_ipython().system('pip install -q cupy-cuda12x')
    import cupy  # noqa: F401
    print("CuPy installato correttamente.")
except Exception as e:
    print(f"CuPy non disponibile ({e}).")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 114.1 MB/s eta 0:00:00
CuPy installato correttamente.


In [3]:
import re, glob

candidates = glob.glob("/usr/**/dist-packages/IPPy/operators.py", recursive=True) + \
             glob.glob("/usr/**/site-packages/IPPy/operators.py", recursive=True)
assert candidates, "Non trovo operators.py di IPPy installato."
operators_path = candidates[0]

with open(operators_path) as f:
    src = f.read()

pattern = re.compile(
    r'(elif torch\.cuda\.is_available\(\) and not force_cpu:\s*\n'
    r'\s*warnings\.warn\(\s*\n'
    r'\s*"CUDA available but CuPy not found\. CTProjector limited to CPU operations for ASTRA data transfer\."\s*\n'
    r'\s*\)\s*\n'
    r'\s*# Force CPU mode if CuPy isn\'t there for GPU data handling\s*\n'
    r'(\s*)self\.use_gpu = False)'
)

def _fix(m):
    indent = m.group(2)
    return (
        "elif torch.cuda.is_available() and not force_cpu:\n"
        f"{indent}try:\n"
        f"{indent}    import cupy  # noqa: F401\n"
        f"{indent}except ImportError:\n"
        f"{indent}    warnings.warn(\n"
        f'{indent}        "CUDA available but CuPy not found. CTProjector limited to CPU operations for ASTRA data transfer."\n'
        f"{indent}    )\n"
        f"{indent}    self.use_gpu = False\n"
    )

new_src, n = pattern.subn(_fix, src)
if n == 1:
    with open(operators_path, "w") as f:
        f.write(new_src)
    print(f"Patch applicata a {operators_path}")
elif "import cupy  # noqa: F401" in src:
    print("Patch gia' presente, salto.")
else:
    raise RuntimeError(f"Blocco da patchare non trovato in {operators_path}.")

Patch applicata a /usr/local/lib/python3.13/dist-packages/IPPy/operators.py


In [4]:
import math
import time
import json

import numpy as np
from skimage import io
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from IPPy import operators
from IPPy.utilities.metrics import PSNR, SSIM

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo in uso: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def load_normalized_image(path: Path) -> np.ndarray:
    image = io.imread(path).astype(np.float32)
    max_value = np.iinfo(io.imread(path).dtype).max
    return image / max_value

Dispositivo in uso: cuda
GPU: NVIDIA A100-SXM4-40GB


In [5]:
# ============================================================
# Generatore (stessa architettura, pesi gia' allenati e fissi)
# ============================================================

class Generator(nn.Module):
    def __init__(self, latent_dim=100, ngf=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 16, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 16), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 16, ngf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 8), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),

            nn.ConvTranspose2d(ngf, ngf // 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf // 2), nn.ReLU(True),

            nn.ConvTranspose2d(ngf // 2, 1, 4, 2, 1, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), self.latent_dim, 1, 1))


LATENT_DIM = 100
NGF = 32

generator = Generator(LATENT_DIM, NGF).to(DEVICE)
generator.load_state_dict(torch.load(GENERATOR_WEIGHTS_PATH, map_location=DEVICE))
generator.eval()
for p in generator.parameters():
    p.requires_grad_(False)

print(f"Generatore caricato da: {GENERATOR_WEIGHTS_PATH}")

Generatore caricato da: /content/drive/MyDrive/gan_checkpoints/generator_weights_best.pth


In [6]:
# ============================================================
# Geometria, proiettori, ciclo DPS -- versione con Adam come
# ottimizzatore per z (non discesa a passo fisso: quella richiedeva
# uno step_size sempre piu' piccolo senza mai stabilizzarsi su un
# vero ottimo, sintomo di gradiente mal scalato tra le componenti di z).
# ============================================================

IMG_SIZE = (256, 256)
NOISE_LEVEL = 0.005

ANGLE_CONFIGS = {
    90: np.linspace(-45, 45, 90),
    45: np.linspace(-45, 45, 45),
    30: np.linspace(-30, 30, 32)[1:-1],
    15: np.linspace(-30, 30, 17)[1:-1],
}

PROJECTORS = {
    n_angles: operators.CTProjector(img_shape=IMG_SIZE, angles=np.deg2rad(angles), geometry="parallel", force_cpu=False)
    for n_angles, angles in ANGLE_CONFIGS.items()
}


def dps_reconstruct(y_delta, K, n_iters, data_weight, step_size, noise_scale):
    z = torch.randn(1, LATENT_DIM, device=DEVICE, requires_grad=True)
    optimizer = torch.optim.Adam([z], lr=step_size)   # step_size = learning rate di Adam

    for _ in range(n_iters):
        optimizer.zero_grad()

        x = generator(z)
        x_cpu = x.cpu()

        residual = K(x_cpu) - y_delta
        fidelity_loss = torch.sum(residual ** 2)
        prior_loss = torch.sum(z ** 2)
        loss = data_weight * fidelity_loss + prior_loss.cpu()

        loss.backward()
        optimizer.step()

        if noise_scale > 0:
            with torch.no_grad():
                z += math.sqrt(2 * step_size) * noise_scale * torch.randn_like(z)

    with torch.no_grad():
        x_final = generator(z)
    return x_final.detach().cpu().numpy()

Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA
Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA
Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA
Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA


In [7]:
# ============================================================
# Griglia di tuning e parametri
# ============================================================

TUNING_N_ITERS = 300   # stesso numero usato in generazione, per evitare disallineamenti
N_VAL_SAMPLES = 12

DATA_WEIGHT_CANDIDATES = [0.01, 0.1, 1.0]
STEP_SIZE_CANDIDATES = [1e-3, 1e-2, 1e-1]
NOISE_SCALE_CANDIDATES = [0.0, 0.02]

PSNR_WEIGHT = 0.5

n_combos = len(DATA_WEIGHT_CANDIDATES) * len(STEP_SIZE_CANDIDATES) * len(NOISE_SCALE_CANDIDATES)
print(f"Combinazioni per configurazione: {n_combos}")
print(f"Valutazioni totali: {n_combos * len(ANGLE_CONFIGS) * N_VAL_SAMPLES}")

Combinazioni per configurazione: 18
Valutazioni totali: 864


In [8]:
# ============================================================
# Pre-caricamento e cache dei dati di validation (sinogramma + GT)
# ============================================================

cached_data = {}

for n_angles in ANGLE_CONFIGS:
    sino_dir = LOCAL_SINOGRAMS_DIR / "validation" / str(n_angles)
    sino_paths = sorted(sino_dir.rglob("*.npy"))[:N_VAL_SAMPLES]

    samples = []
    for sino_path in sino_paths:
        rel_path = sino_path.relative_to(sino_dir)
        gt_path = (VALIDATION_DIR / rel_path).with_suffix(".png")

        sinogram = np.load(sino_path)
        y_delta = torch.from_numpy(sinogram).float().unsqueeze(0).unsqueeze(0)   # CPU

        gt = load_normalized_image(gt_path)
        x_true = torch.from_numpy(gt).float().unsqueeze(0).unsqueeze(0)

        samples.append((y_delta, x_true))

    cached_data[n_angles] = samples
    print(f"{n_angles} angoli: {len(samples)} campioni in cache")

90 angoli: 12 campioni in cache
45 angoli: 12 campioni in cache
30 angoli: 12 campioni in cache
15 angoli: 12 campioni in cache


In [9]:
# ============================================================
# Test rapido su UNA combinazione: stima il tempo totale della griglia
# ============================================================

test_n_angles = 90
y_delta, x_true = cached_data[test_n_angles][0]

torch.manual_seed(123)
t0 = time.time()
_ = dps_reconstruct(
    y_delta, PROJECTORS[test_n_angles],
    n_iters=TUNING_N_ITERS,
    data_weight=DATA_WEIGHT_CANDIDATES[0], step_size=STEP_SIZE_CANDIDATES[0], noise_scale=NOISE_SCALE_CANDIDATES[0],
)
elapsed = time.time() - t0

n_valutazioni_totali = n_combos * len(ANGLE_CONFIGS) * N_VAL_SAMPLES
print(f"Tempo per una valutazione ({TUNING_N_ITERS} iterazioni): {elapsed:.2f} s")
print(f"Stima tempo totale griglia ({n_valutazioni_totali} valutazioni): {elapsed * n_valutazioni_totali / 60:.1f} minuti")

Tempo per una valutazione (300 iterazioni): 2.67 s
Stima tempo totale griglia (864 valutazioni): 38.4 minuti


In [10]:
# ============================================================
# GRID SEARCH: data_weight x step_size x noise_scale, per configurazione
# ============================================================

grid_results = {}

for n_angles in ANGLE_CONFIGS:
    print(f"\n=== Tuning per {n_angles} angoli ===")
    samples = cached_data[n_angles]
    K = PROJECTORS[n_angles]

    combos = [
        (dw, ss, ns)
        for dw in DATA_WEIGHT_CANDIDATES
        for ss in STEP_SIZE_CANDIDATES
        for ns in NOISE_SCALE_CANDIDATES
    ]

    results = []
    for data_weight, step_size, noise_scale in tqdm(combos, desc=f"[{n_angles} angoli] griglia"):
        psnr_list, ssim_list = [], []

        for y_delta, x_true in samples:
            torch.manual_seed(123)
            x_sol = dps_reconstruct(
                y_delta, K, n_iters=TUNING_N_ITERS,
                data_weight=data_weight, step_size=step_size, noise_scale=noise_scale,
            )
            x_sol_clipped = np.clip(x_sol, 0.0, 1.0)

            x_pred = torch.from_numpy(x_sol_clipped).float()
            psnr_list.append(float(PSNR(x_pred, x_true)))
            ssim_list.append(float(SSIM(x_pred, x_true)))

        results.append({
            "data_weight": data_weight, "step_size": step_size, "noise_scale": noise_scale,
            "psnr": float(np.mean(psnr_list)), "ssim": float(np.mean(ssim_list)),
        })

    grid_results[n_angles] = results
    print(f"Completato: {len(results)} combinazioni testate per {n_angles} angoli.")


=== Tuning per 90 angoli ===


[90 angoli] griglia:   0%|          | 0/18 [00:00<?, ?it/s]

Completato: 18 combinazioni testate per 90 angoli.

=== Tuning per 45 angoli ===


[45 angoli] griglia:   0%|          | 0/18 [00:00<?, ?it/s]

Completato: 18 combinazioni testate per 45 angoli.

=== Tuning per 30 angoli ===


[30 angoli] griglia:   0%|          | 0/18 [00:00<?, ?it/s]

Completato: 18 combinazioni testate per 30 angoli.

=== Tuning per 15 angoli ===


[15 angoli] griglia:   0%|          | 0/18 [00:00<?, ?it/s]

Completato: 18 combinazioni testate per 15 angoli.


In [11]:
# ============================================================
# Selezione della combinazione migliore per configurazione + salvataggio
# -- UNICO file di output, quello che 10_GAN_DPS_generation.ipynb legge
# ============================================================

best_params_per_config = {}

TUNING_DIR = Path("/content/drive/MyDrive/gan_dps_tuning")
TUNING_DIR.mkdir(parents=True, exist_ok=True)

for n_angles, results in grid_results.items():
    psnr_values = np.array([r["psnr"] for r in results])
    ssim_values = np.array([r["ssim"] for r in results])

    psnr_norm = (psnr_values - psnr_values.min()) / (psnr_values.max() - psnr_values.min() + 1e-12)
    ssim_norm = (ssim_values - ssim_values.min()) / (ssim_values.max() - ssim_values.min() + 1e-12)
    combined = PSNR_WEIGHT * psnr_norm + (1 - PSNR_WEIGHT) * ssim_norm

    order = np.argsort(-combined)

    print(f"\n=== {n_angles} angoli -- top 5 combinazioni ===")
    for idx in order[:5]:
        r = results[idx]
        print(f"  data_weight={r['data_weight']:<6} step_size={r['step_size']:<7} noise_scale={r['noise_scale']:<5} "
              f"-> PSNR={r['psnr']:.2f} dB  SSIM={r['ssim']:.4f}  score={combined[idx]:.4f}")

    best_idx = int(order[0])
    best_params_per_config[n_angles] = results[best_idx]

print("\n=== Riepilogo: parametri scelti per configurazione ===")
for n_angles, params in best_params_per_config.items():
    print(f"{n_angles} angoli -> data_weight={params['data_weight']}, step_size={params['step_size']}, noise_scale={params['noise_scale']}")

with open(TUNING_DIR / "best_params_dps.json", "w") as f:
    json.dump(best_params_per_config, f, indent=2)

with open(TUNING_DIR / "full_grid_results_dps.json", "w") as f:
    json.dump(grid_results, f, indent=2)

print(f"\nSalvato in: {TUNING_DIR}")


=== 90 angoli -- top 5 combinazioni ===
  data_weight=0.01   step_size=0.01    noise_scale=0.0   -> PSNR=17.44 dB  SSIM=0.3274  score=0.9802
  data_weight=0.1    step_size=0.01    noise_scale=0.0   -> PSNR=17.44 dB  SSIM=0.3274  score=0.9801
  data_weight=1.0    step_size=0.01    noise_scale=0.0   -> PSNR=17.44 dB  SSIM=0.3274  score=0.9799
  data_weight=1.0    step_size=0.001   noise_scale=0.0   -> PSNR=17.32 dB  SSIM=0.3317  score=0.9703
  data_weight=0.1    step_size=0.001   noise_scale=0.0   -> PSNR=17.32 dB  SSIM=0.3317  score=0.9700

=== 45 angoli -- top 5 combinazioni ===
  data_weight=0.01   step_size=0.01    noise_scale=0.0   -> PSNR=17.44 dB  SSIM=0.3277  score=0.9803
  data_weight=1.0    step_size=0.01    noise_scale=0.0   -> PSNR=17.44 dB  SSIM=0.3276  score=0.9799
  data_weight=0.1    step_size=0.01    noise_scale=0.0   -> PSNR=17.44 dB  SSIM=0.3275  score=0.9791
  data_weight=0.01   step_size=0.001   noise_scale=0.0   -> PSNR=17.33 dB  SSIM=0.3318  score=0.9711
  data_we

## Prossimo passo
`10_GAN_DPS_generation.ipynb` legge automaticamente `gan_dps_tuning/best_params_dps.json` -- non serve copiare nulla a mano. Verifica solo che `GENERATOR_WEIGHTS_PATH` in quel notebook punti a `gan_checkpoints/generator_weights_best.pth` (il generatore senza decay, FID 314.77 -- non `gan_checkpoints_decay`).